# Mathematical notes for `QuadraticNumberFields/ClassGroup/`

This notebook explains the mathematics corresponding to the folder
`QuadraticNumberFields/ClassGroup/`. It is not just a commentary on one Lean proof.

The main line of the folder is:

```text
class group / class number
        ↓
Minkowski: every ideal class has a representative of small norm
        ↓
it is enough to inspect finitely many small rational primes
        ↓
use the split / inert / ramified behavior above those primes
        ↓
class number = 1
```

The source modules play these roles:

- `ClassNumber.lean`: connects the general number-field class number to `Qsqrtd d`.
- `Minkowski.lean`: specializes the Minkowski bound to quadratic fields `Q(sqrt(d))` and turns splitting behavior of small primes into class-number-one criteria.
- `SmallNorm.lean`: if every class has a representative of norm `< 3`, only norms `1` and `2` remain.


In [1]:
import QuadraticNumberFields.ClassGroup.Minkowski
import QuadraticNumberFields.ClassGroup.SmallNorm
import QuadraticNumberFields.RingOfIntegers.CommonInstances
import QuadraticNumberFields.Splitting.Qsqrtd.Classification

attribute [-instance] DivisionRing.toRatAlgebra

open scoped NumberField Real
open scoped QuadraticNumberFields.Splitting

namespace QuadraticNumberFields
namespace ClassNumberExample


import QuadraticNumberFields.ClassGroup.Minkowski
import QuadraticNumberFields.ClassGroup.SmallNorm
import QuadraticNumberFields.RingOfIntegers.CommonInstances
import QuadraticNumberFields.Splitting.Qsqrtd.Classification

attribute [-instance] DivisionRing.toRatAlgebra

open scoped NumberField Real
open scoped QuadraticNumberFields.Splitting

namespace QuadraticNumberFields
namespace ClassNumberExample

--% env 0

Raw input:
{"cmd": "import QuadraticNumberFields.ClassGroup.Minkowski\nimport QuadraticNumberFields.ClassGroup.SmallNorm\nimport QuadraticNumberFields.RingOfIntegers.CommonInstances\nimport QuadraticNumberFields.Splitting.Qsqrtd.Classification\n\nattribute [-instance] DivisionRing.toRatAlgebra\n\nopen scoped NumberField Real\nopen scoped QuadraticNumberFields.Splitting\n\nnamespace QuadraticNumberFields\nnamespace ClassNumberExample\n"}
Raw output:
{"env": 0}

## 1. `ClassNumber.lean`: the class number is the size of the class group

The mathematical objects are:

- `𝓞 K`: the ring of integers of a number field `K`.
- `ClassGroup (𝓞 K)`: the ideal class group of the Dedekind domain `𝓞 K`.
- `NumberField.classNumber K`: the cardinality of this class group.

For quadratic fields, this project mainly uses the standard coordinate model:

```lean
Qsqrtd (d : Q) = Q(sqrt(d))
```

So `classNumberQsqrtd d` is just:

```lean
NumberField.classNumber (Qsqrtd (d : Q))
```

The other basic theorem in `ClassNumber.lean` says: if every ideal class is `1`, then the
class number is `1`. This is the Lean entry point for the mathematical implication
"trivial class group implies class number one."


In [2]:
% env 0
#check classNumberQsqrtd
#check NumberField.classNumber
#check ClassGroup
#check NumberField.classNumber_eq_one_of_forall_classGroup_eq_one


--% env 0
#check classNumberQsqrtd
──────▶  QuadraticNumberFields.classNumberQsqrtd (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] : ℕ
#check NumberField.classNumber
──────▶  NumberField.classNumber.{u_1} (K : Type u_1) [Field K] [NumberField K] : ℕ
#check ClassGroup
──────▶  ClassGroup.{u_1} (R : Type u_1) [CommRing R] [IsDomain R] : Type u_1
#check NumberField.classNumber_eq_one_of_forall_classGroup_eq_one
──────▶  QuadraticNumberFields.NumberField.classNumber_eq_one_of_forall_classGroup_eq_one.{u_1} {K : Type u_1} [Field K]
  [NumberField K] (h : ∀ (C : ClassGroup (𝓞 K)), C = 1) : NumberField.classNumber K = 1

--% env 1

Raw input:
{"cmd": "--% env 0\n#check classNumberQsqrtd\n#check NumberField.classNumber\n#check ClassGroup\n#check NumberField.classNumber_eq_one_of_forall_classGroup_eq_one\n", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "QuadraticNumberFields.classNumberQsqrtd (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] : ℕ"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "NumberField.classNumber.{u_1} (K : Type u_1) [Field K] [NumberField K] : ℕ"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "ClassGroup.{u_1} (R : Type u_1) [CommRing R] [IsDomain R] : Type u_1"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "QuadraticNumberFields.NumberField.classNumber_eq_one_of_forall_classGroup_eq_one.{u_1} {K : Type u_1} [Field K]\n  [NumberField K] (h : ∀ (C : ClassGroup (𝓞 K)), C = 1) : NumberField.classNumber K = 1"}],
 "env": 1}

## 2. `Minkowski.lean`: the Minkowski bound for quadratic fields

Minkowski's theorem says that every ideal class has an integral ideal representative
`I` whose absolute norm `absNorm I` is bounded by an explicit constant.

For a general number field, the constant is:

```text
(4 / pi)^r2 * (n! / n^n) * sqrt(|D_K|)
```

For a quadratic field `Q(sqrt(d))`, the degree is `n = 2`, so `n! / n^n = 2 / 4 = 1 / 2`.
Thus:

- imaginary quadratic, `d < 0`: there is one complex place, so the bound is `(2 / pi) * sqrt(|D_K|)`;
- real quadratic, `0 < d`: there are no complex places, so the bound is `(1 / 2) * sqrt(|D_K|)`.

The source file packages this uniformly as:

```lean
Qsqrtd.minkowskiBound d
```


In [3]:
#check Qsqrtd.nrComplexPlaces_eq_one_of_neg
#check Qsqrtd.nrComplexPlaces_eq_zero_of_pos
#check Qsqrtd.minkowskiBound
#check Qsqrtd.exists_ideal_in_class_of_norm_le
#check Qsqrtd.exists_ideal_in_class_of_norm_le_imaginary
#check Qsqrtd.exists_ideal_in_class_of_norm_le_real


#check Qsqrtd.nrComplexPlaces_eq_one_of_neg
──────▶  QuadraticNumberFields.Qsqrtd.nrComplexPlaces_eq_one_of_neg (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] (hd : d < 0) :
  NumberField.InfinitePlace.nrComplexPlaces (ℚ√↑d) = 1
#check Qsqrtd.nrComplexPlaces_eq_zero_of_pos
──────▶  QuadraticNumberFields.Qsqrtd.nrComplexPlaces_eq_zero_of_pos (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] (hd : 0 < d) :
  NumberField.InfinitePlace.nrComplexPlaces (ℚ√↑d) = 0
#check Qsqrtd.minkowskiBound
──────▶  QuadraticNumberFields.Qsqrtd.minkowskiBound (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] : ℝ
#check Qsqrtd.exists_ideal_in_class_of_norm_le
──────▶  QuadraticNumberFields.Qsqrtd.exists_ideal_in_class_of_norm_le (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)]
  (C : ClassGroup 𝓞(↑d)) : ∃ I, ClassGroup.mk0 I = C ∧ ↑(Ideal.absNorm ↑I) ≤ Qsqrtd.minkowskiBound d
#check Qsqrtd.exists_ideal_in_class_of_norm_le_imaginary
──────▶  QuadraticNumberFields.Qsqrtd.exists_ideal_in_class_of_norm_le_imaginary (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)]
  (hd : d < 0) (C : ClassGroup 𝓞(↑d)) :
  ∃ I, ClassGroup.mk0 I = C ∧ ↑(Ideal.absNorm ↑I) ≤ 2 / π * √|↑(NumberField.discr (ℚ√↑d))|
#check Qsqrtd.exists_ideal_in_class_of_norm_le_real
──────▶  QuadraticNumberFields.Qsqrtd.exists_ideal_in_class_of_norm_le_real (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)]
  (hd : 0 < d) (C : ClassGroup 𝓞(↑d)) :
  ∃ I, ClassGroup.mk0 I = C ∧ ↑(Ideal.absNorm ↑I) ≤ 1 / 2 * √|↑(NumberField.discr (ℚ√↑d))|

--% env 2

Raw input:
{"cmd": "#check Qsqrtd.nrComplexPlaces_eq_one_of_neg\n#check Qsqrtd.nrComplexPlaces_eq_zero_of_pos\n#check Qsqrtd.minkowskiBound\n#check Qsqrtd.exists_ideal_in_class_of_norm_le\n#check Qsqrtd.exists_ideal_in_class_of_norm_le_imaginary\n#check Qsqrtd.exists_ideal_in_class_of_norm_le_real\n", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.nrComplexPlaces_eq_one_of_neg (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] (hd : d < 0) :\n  NumberField.InfinitePlace.nrComplexPlaces (ℚ√↑d) = 1"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.nrComplexPlaces_eq_zero_of_pos (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] (hd : 0 < d) :\n  NumberField.InfinitePlace.nrComplexPlaces (ℚ√↑d) = 0"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.minkowskiBound (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] : ℝ"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.exists_ideal_in_class_of_norm_le (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)]\n  (C : ClassGroup 𝓞(↑d)) : ∃ I, ClassGroup.mk0 I = C ∧ ↑(Ideal.absNorm ↑I) ≤ Qsqrtd.minkowskiBound d"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.exists_ideal_in_class_of_norm_le_imaginary (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)]\n  (hd : d < 0) (C : ClassGroup 𝓞(↑d)) :\n  ∃ I, ClassGroup.mk0 I = C ∧ ↑(Ideal.absNorm ↑I) ≤ 2 / π * √|↑(NumberField.discr (ℚ√↑d))|"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.exists_ideal_in_class_of_norm_le_real (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)]\n  (hd : 0 < d) (C : ClassGroup 𝓞(↑d)) :\n  ∃ I, ClassGroup.mk0 I = C ∧ ↑(Ideal.absNorm ↑I) ≤ 1 / 2 * √|↑(NumberField.discr (ℚ√↑d))|"}],
 "env": 2}

### Numeric versions of the bound

For concrete examples, we do not want to redo real-number estimates by hand each time.
`Minkowski.lean` provides two versions of the statement "an integer discriminant
inequality implies `minkowskiBound d < n`."

Imaginary case: if

```text
4 * |D_K| < 9 * n^2
```

then

```text
minkowskiBound d < n
```

The factor `9` comes from the rough estimate `pi > 3`. It is not sharp, but it is
strong enough for the small computations in the Heegner class-number-one proofs.

Real case: if

```text
|D_K| < 4 * n^2
```

then

```text
minkowskiBound d < n
```


In [4]:
#check Qsqrtd.minkowskiBound_lt_of_neg
#check Qsqrtd.minkowskiBound_lt_of_pos

example : Qsqrtd.minkowskiBound (-1 : ℤ) < (2 : ℕ) := by
  exact Qsqrtd.minkowskiBound_lt_of_neg _ (by norm_num)
    (by rw [RingOfIntegers.discr_of_mod_four_ne_one _ (by decide)]; norm_num)


#check Qsqrtd.minkowskiBound_lt_of_neg
──────▶  QuadraticNumberFields.Qsqrtd.minkowskiBound_lt_of_neg (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] (hd : d < 0) {n : ℕ}
  (hn : 4 * |NumberField.discr (ℚ√↑d)| < 9 * ↑n ^ 2) : Qsqrtd.minkowskiBound d < ↑n
#check Qsqrtd.minkowskiBound_lt_of_pos
──────▶  QuadraticNumberFields.Qsqrtd.minkowskiBound_lt_of_pos (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] (hd : 0 < d) {n : ℕ}
  (hn : |NumberField.discr (ℚ√↑d)| < 4 * ↑n ^ 2) : Qsqrtd.minkowskiBound d < ↑n

example : Qsqrtd.minkowskiBound (-1 : ℤ) < (2 : ℕ) := by
  exact Qsqrtd.minkowskiBound_lt_of_neg _ (by norm_num)
    (by rw [RingOfIntegers.discr_of_mod_four_ne_one _ (by decide)]; norm_num)

--% env 3

Raw input:
{"cmd": "#check Qsqrtd.minkowskiBound_lt_of_neg\n#check Qsqrtd.minkowskiBound_lt_of_pos\n\nexample : Qsqrtd.minkowskiBound (-1 : \u2124) < (2 : \u2115) := by\n  exact Qsqrtd.minkowskiBound_lt_of_neg _ (by norm_num)\n    (by rw [RingOfIntegers.discr_of_mod_four_ne_one _ (by decide)]; norm_num)\n", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.minkowskiBound_lt_of_neg (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] (hd : d < 0) {n : ℕ}\n  (hn : 4 * |NumberField.discr (ℚ√↑d)| < 9 * ↑n ^ 2) : Qsqrtd.minkowskiBound d < ↑n"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.minkowskiBound_lt_of_pos (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] (hd : 0 < d) {n : ℕ}\n  (hn : |NumberField.discr (ℚ√↑d)| < 4 * ↑n ^ 2) : Qsqrtd.minkowskiBound d < ↑n"}],
 "env": 3}

## 3. `SmallNorm.lean`: when the norm is `< 3`, only two cases remain

If a nonzero integral ideal `I` satisfies:

```text
absNorm I < 3
```

then, because the norm is a positive natural number, only two possibilities remain:

```text
absNorm I = 1  or  absNorm I = 2
```

- `absNorm I = 1`: the ideal is the whole ring, hence principal.
- `absNorm I = 2`: one still needs to prove that all ideals of norm `2` are principal,
  or at least that they all represent a fixed ideal class `P`.

So `SmallNorm.lean` is a compression layer: it reduces a class-group question to a
question about ideals of norm `2`.


In [ ]:
#check Ideal.isPrime_of_absNorm_eq_two
#check classGroup_eq_one_of_exists_ideal_norm_lt_three
#check classGroup_eq_one_or_of_exists_ideal_norm_lt_three


## 4. `Minkowski.lean`: from small-prime splitting behavior to class number one

Minkowski gives this information: every class has an ideal representative of small norm.

To prove that this representative is principal, decompose it into prime-ideal factors.
Every prime ideal `P` lies over some rational prime `p`, and the size of `p` is bounded
by `absNorm P`, then by `absNorm I`, then by the Minkowski bound.

Therefore, it is enough to inspect:

```text
all rational primes p with p <= minkowskiBound d
```

`Minkowski.lean` gives several versions:

1. all small primes are inert ⇒ class number one;
2. every prime ideal above every small prime is principal ⇒ class number one;
3. small primes are inert or split, and the split fibers are principal ⇒ class number one;
4. small primes are inert or ramified, and the ramified fibers are principal ⇒ class number one.

The key mathematical point: if `(p)` is inert, then `(p)` remains prime in the ring
of integers. Hence the prime ideal above `(p)` is just `(p)` itself, which is principal.


In [6]:
#check Qsqrtd.classGroup_eq_one_of_forall_le_minkowskiBound_isInertIn
#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_isInertIn
#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_primesOver_isPrincipal
#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_split_principal
#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_ramified_principal


#check Qsqrtd.classGroup_eq_one_of_forall_le_minkowskiBound_isInertIn
──────▶  QuadraticNumberFields.Qsqrtd.classGroup_eq_one_of_forall_le_minkowskiBound_isInertIn (d : ℤ) [Fact (Squarefree d)]
  [Fact (d ≠ 1)] (h : ∀ (p : ℕ), Nat.Prime p → ↑p ≤ Qsqrtd.minkowskiBound d → 𝔭(↑p).IsInertIn 𝓞(↑d))
  (C : ClassGroup 𝓞(↑d)) : C = 1
#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_isInertIn
──────▶  QuadraticNumberFields.Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_isInertIn (d : ℤ) [Fact (Squarefree d)]
  [Fact (d ≠ 1)] (h : ∀ (p : ℕ), Nat.Prime p → ↑p ≤ Qsqrtd.minkowskiBound d → 𝔭(↑p).IsInertIn 𝓞(↑d)) :
  NumberField.classNumber (ℚ√↑d) = 1
#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_primesOver_isPrincipal
──────▶  QuadraticNumberFields.Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_primesOver_isPrincipal (d : ℤ)
  [Fact (Squarefree d)] [Fact (d ≠ 1)]
  (h : ∀ (p : ℕ), Nat.Prime p → ↑p ≤ Qsqrtd.minkowskiBound d → ∀ P ∈ 𝔭(↑p).primesOver 𝓞(↑d), Submodule.IsPrincipal P) :
  NumberField.classNumber (ℚ√↑d) = 1
#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_split_principal
──────▶  QuadraticNumberFields.Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_split_principal (d : ℤ)
  [Fact (Squarefree d)] [Fact (d ≠ 1)]
  (hinert_or_split :
    ∀ (p : ℕ), Nat.Prime p → ↑p ≤ Qsqrtd.minkowskiBound d → 𝔭(↑p).IsInertIn 𝓞(↑d) ∨ 𝔭(↑p).IsSplitIn 𝓞(↑d))
  (hsplit :
    ∀ (p : ℕ),
      Nat.Prime p →
        ↑p ≤ Qsqrtd.minkowskiBound d → 𝔭(↑p).IsSplitIn 𝓞(↑d) → ∀ P ∈ 𝔭(↑p).primesOver 𝓞(↑d), Submodule.IsPrincipal P) :
  NumberField.classNumber (ℚ√↑d) = 1
#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_ramified_principal
──────▶  QuadraticNumberFields.Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_ramified_principal (d : ℤ)
  [Fact (Squarefree d)] [Fact (d ≠ 1)]
  (hinert_or_ramified :
    ∀ (p : ℕ), Nat.Prime p → ↑p ≤ Qsqrtd.minkowskiBound d → 𝔭(↑p).IsInertIn 𝓞(↑d) ∨ 𝔭(↑p).IsRamifiedIn 𝓞(↑d))
  (hramified :
    ∀ (p : ℕ),
      Nat.Prime p →
        ↑p ≤ Qsqrtd.minkowskiBound d →
          𝔭(↑p).IsRamifiedIn 𝓞(↑d) → ∀ P ∈ 𝔭(↑p).primesOver 𝓞(↑d), Submodule.IsPrincipal P) :
  NumberField.classNumber (ℚ√↑d) = 1

--% env 5

Raw input:
{"cmd": "#check Qsqrtd.classGroup_eq_one_of_forall_le_minkowskiBound_isInertIn\n#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_isInertIn\n#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_primesOver_isPrincipal\n#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_split_principal\n#check Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_ramified_principal\n", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.classGroup_eq_one_of_forall_le_minkowskiBound_isInertIn (d : ℤ) [Fact (Squarefree d)]\n  [Fact (d ≠ 1)] (h : ∀ (p : ℕ), Nat.Prime p → ↑p ≤ Qsqrtd.minkowskiBound d → 𝔭(↑p).IsInertIn 𝓞(↑d))\n  (C : ClassGroup 𝓞(↑d)) : C = 1"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_isInertIn (d : ℤ) [Fact (Squarefree d)]\n  [Fact (d ≠ 1)] (h : ∀ (p : ℕ), Nat.Prime p → ↑p ≤ Qsqrtd.minkowskiBound d → 𝔭(↑p).IsInertIn 𝓞(↑d)) :\n  NumberField.classNumber (ℚ√↑d) = 1"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_primesOver_isPrincipal (d : ℤ)\n  [Fact (Squarefree d)] [Fact (d ≠ 1)]\n  (h : ∀ (p : ℕ), Nat.Prime p → ↑p ≤ Qsqrtd.minkowskiBound d → ∀ P ∈ 𝔭(↑p).primesOver 𝓞(↑d), Submodule.IsPrincipal P) :\n  NumberField.classNumber (ℚ√↑d) = 1"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "QuadraticNumberFields.Qsqrtd.classNumber_eq_o

## 5. Worked example: `Q(sqrt(-1))`

Now we run the whole strategy in one concrete case.

Goal:

```text
classNumber(Q(sqrt(-1))) = 1
```

Steps:

1. Use the inert-prime criterion: it is enough to prove that every rational prime
   `p <= minkowskiBound(-1)` is inert.
2. Show that `minkowskiBound(-1) < 2`.
3. Hence `p < 2`, so `p` can only be `0` or `1`.
4. Neither `0` nor `1` is prime, so there are no rational primes to check. The
   inertness condition is vacuously true.

Mathematically, the discriminant of `Q(sqrt(-1))` is `-4`, and the imaginary
quadratic bound is:

```text
(2 / pi) * sqrt(4) = 4 / pi < 2
```


In [7]:
/-- Learning version: `Q(sqrt(-1))` has class number one by the Minkowski criterion. -/
theorem classNumber_eq_one_neg1_from_minkowski :
    NumberField.classNumber (Qsqrtd ((-1 : ℤ) : ℚ)) = 1 := by
  refine Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_isInertIn (-1)
    fun p hp hple => ?_
  have hb : Qsqrtd.minkowskiBound (-1 : ℤ) < (2 : ℕ) :=
    Qsqrtd.minkowskiBound_lt_of_neg _ (by norm_num)
      (by rw [RingOfIntegers.discr_of_mod_four_ne_one _ (by decide)]; norm_num)
  have hplt : p < 2 := by exact_mod_cast hple.trans_lt hb
  interval_cases p <;> exact absurd hp (by decide)


/-- Learning version: `Q(sqrt(-1))` has class number one by the Minkowski criterion. -/
theorem classNumber_eq_one_neg1_from_minkowski :
    NumberField.classNumber (Qsqrtd ((-1 : ℤ) : ℚ)) = 1 := by
  refine Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_isInertIn (-1)
    fun p hp hple => ?_
  have hb : Qsqrtd.minkowskiBound (-1 : ℤ) < (2 : ℕ) :=
    Qsqrtd.minkowskiBound_lt_of_neg _ (by norm_num)
      (by rw [RingOfIntegers.discr_of_mod_four_ne_one _ (by decide)]; norm_num)
  have hplt : p < 2 := by exact_mod_cast hple.trans_lt hb
  interval_cases p <;> exact absurd hp (by decide)

--% env 6

Raw input:
{"cmd": "/-- Learning version: `Q(sqrt(-1))` has class number one by the Minkowski criterion. -/\ntheorem classNumber_eq_one_neg1_from_minkowski :\n    NumberField.classNumber (Qsqrtd ((-1 : \u2124) : \u211a)) = 1 := by\n  refine Qsqrtd.classNumber_eq_one_of_forall_le_minkowskiBound_isInertIn (-1)\n    fun p hp hple => ?_\n  have hb : Qsqrtd.minkowskiBound (-1 : \u2124) < (2 : \u2115) :=\n    Qsqrtd.minkowskiBound_lt_of_neg _ (by norm_num)\n      (by rw [RingOfIntegers.discr_of_mod_four_ne_one _ (by decide)]; norm_num)\n  have hplt : p < 2 := by exact_mod_cast hple.trans_lt hb\n  interval_cases p <;> exact absurd hp (by decide)\n", "env": 5}
Raw output:
{"env": 6}

## 6. How this scales beyond `d = -1`

For larger negative `d`, the Minkowski bound may be less than `3`, `5`, `6`, or `9`
rather than less than `2`. Then the proof is no longer vacuous: one must check
actual small primes.

For example, for `d = -163` the bound is `< 9`, so the relevant rational primes are:

```text
2, 3, 5, 7
```

The Heegner application layer proves these inertness checks using the splitting
classification:

- `p = 2`: use congruences modulo `8`;
- odd `p`: use Legendre-symbol / Kronecker-symbol criteria.

This folder supplies the class-number machinery. The concrete Heegner theorem file
supplies the finite splitting computations.


In [8]:
#check Splitting.isInert_two_of_mod_eight_eq_five
#check Splitting.isInert_iff_legendreSym_eq_neg_one

end ClassNumberExample
end QuadraticNumberFields


#check Splitting.isInert_two_of_mod_eight_eq_five
──────▶  QuadraticNumberFields.Splitting.isInert_two_of_mod_eight_eq_five (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)]
  (hd8 : d % 8 = 5) : 𝔭(2).IsInertIn 𝓞(↑d)
#check Splitting.isInert_iff_legendreSym_eq_neg_one
──────▶  QuadraticNumberFields.Splitting.isInert_iff_legendreSym_eq_neg_one (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] (p : ℕ)
  [Fact (Nat.Prime p)] (hp : p ≠ 2) (hpd : ¬↑p ∣ d) : 𝔭(↑p).IsInertIn 𝓞(↑d) ↔ legendreSym p d = -1

end ClassNumberExample
end QuadraticNumberFields

--% env 7

Raw input:
{"cmd": "#check Splitting.isInert_two_of_mod_eight_eq_five\n#check Splitting.isInert_iff_legendreSym_eq_neg_one\n\nend ClassNumberExample\nend QuadraticNumberFields\n", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "QuadraticNumberFields.Splitting.isInert_two_of_mod_eight_eq_five (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)]\n  (hd8 : d % 8 = 5) : 𝔭(2).IsInertIn 𝓞(↑d)"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "QuadraticNumberFields.Splitting.isInert_iff_legendreSym_eq_neg_one (d : ℤ) [Fact (Squarefree d)] [Fact (d ≠ 1)] (p : ℕ)\n  [Fact (Nat.Prime p)] (hp : p ≠ 2) (hpd : ¬↑p ∣ d) : 𝔭(↑p).IsInertIn 𝓞(↑d) ↔ legendreSym p d = -1"}],
 "env": 7}

## 7. Lean4 Jupyter workflow: repeating cells and trying tactics

`lean4_jupyter` has two separate kinds of state.

First, there is the **environment state**. It contains commands already accepted by Lean: imports, definitions, theorems, instances, namespaces, and so on. After a cell runs, the kernel output usually shows something like:

```text
--% env 12
```

That means "the Lean environment after this cell is environment `12`". To go back to an earlier environment, use:

```text
% env 12
```

This is useful when you defined a named theorem and want to rerun that cell without getting a duplicate-name error.

Second, there is the **proof state**. If a theorem contains `sorry`, the output usually shows something like:

```text
--% prove 3
```

That number names the goal at the `sorry`. You can return to that goal and try tactics with:

```text
% prove 3
exact ...
```

If you want the most recent `sorry`, use `% proof` without a number.


### 7.1 Repeatable cells: prefer `example`

For ordinary practice, use `example` instead of a named `theorem`. It creates no permanent declaration name, so it is safe to run the same cell repeatedly with `Ctrl+Enter`.


In [9]:
example {x y : Nat} : x + y = y + x := by
  exact Nat.add_comm x y


example {x y : Nat} : x + y = y + x := by
  exact Nat.add_comm x y

--% env 8

Raw input:
{"cmd": "example {x y : Nat} : x + y = y + x := by\n  exact Nat.add_comm x y\n", "env": 7}
Raw output:
{"env": 8}

### 7.2 Proof-state practice with `% proof`

Run the next cell first. It deliberately leaves a `sorry`, so the kernel should print a proof-state marker such as `--% prove N`.


In [10]:
theorem notebook_add_comm_demo {x y : Nat} : x + y = y + x := sorry


theorem notebook_add_comm_demo {x y : Nat} : x + y = y + x := sorry
        ──────────────────────▶ 🟨 declaration uses `sorry`

--% env 9
--% prove 0

Raw input:
{"cmd": "theorem notebook_add_comm_demo {x y : Nat} : x + y = y + x := sorry\n", "env": 8}
Raw output:
{"sorries":
 [{"proofState": 0,
   "pos": {"line": 1, "column": 62},
   "goal": "x y : ℕ\n⊢ x + y = y + x",
   "endPos": {"line": 1, "column": 67}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 1, "column": 8},
   "endPos": {"line": 1, "column": 30},
   "data": "declaration uses `sorry`"}],
 "env": 9}

Immediately after running the `sorry` cell, run this cell. `% proof` tells the kernel: "enter tactic mode for the most recent `sorry` proof state".

If you want to return to a specific proof state later, replace `% proof` by the number shown in the output, for example:

```text
% prove 3
exact Nat.add_comm x y
```

The actual number depends on the history of your notebook session.


In [11]:
% proof
exact Nat.add_comm x y


--% proof
exact Nat.add_comm x y

▶  Goals accomplished! 🐙
--% env 9
--% prove 1

Raw input:
{"tactic": "--% proof\nexact Nat.add_comm x y\n", "proofState": 0}
Raw output:
{"proofStatus": "Completed", "proofState": 1, "goals": []}

### 7.3 How to recover when rerunning named cells

If you rerun the `notebook_add_comm_demo` cell, Lean may complain that the name already exists. Use the environment marker printed before that theorem was declared.

Example pattern:

```text
% env 12
```

Then rerun the theorem cell. The exact number is notebook-session dependent, so copy it from the output near the cell you want to go back to.

Practical rule:

- For scratch proof attempts, use `example`.
- For learning how tactics solve a goal, use `theorem ... := sorry` followed by `% proof` or `% prove N`.
- For rerunning named declarations, use `% env N` to backtrack before the declaration.


## Summary map

```text
ClassNumber.lean
  class number = cardinality of the ideal class group
  trivial class group -> class number one

Minkowski.lean
  each class has an ideal representative with bounded norm
  in degree 2 the bound becomes explicit from the discriminant

SmallNorm.lean
  if the bound is < 3, only norm 1 and norm 2 representatives matter

Minkowski.lean
  reduce class-number-one proofs to checking small rational primes
  inert primes are automatically harmless because their lifted ideal is principal
```

Use this notebook as the reading order for the folder: first understand the mathematical reduction, then inspect the corresponding `#check` statements, then study the worked `d = -1` proof.
